In [1]:

from pathlib import Path
import sys
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "ml" / "src"))

from aeroguard_ml.data_loader import load_data
from aeroguard_ml.feature_engineering import engineer_features

# Load the original NASA datasets
train_raw, test_raw, rul_df = load_data()

# Load the previously selected feature specification
feature_columns = (
    PROJECT_ROOT / "docs" / "feature-columns.txt"
).read_text(encoding="utf-8-sig").splitlines()

active_sensors = [
    f"sensor_{i}" for i in range(1, 22)
    if f"sensor_{i}" in feature_columns
]

assert len(feature_columns) == 123
assert len(active_sensors) == 15

print("===== FINAL EVALUATION DATA =====")
print("Training engines:", train_raw.engine_id.nunique())
print("Test engines:", test_raw.engine_id.nunique())
print("Model features:", len(feature_columns))
print("Active sensors:", len(active_sensors))


===== FINAL EVALUATION DATA =====
Training engines: 100
Test engines: 100
Model features: 123
Active sensors: 15


In [2]:

# Create RUL targets for complete training engines
full_train = train_raw.copy()

full_train["RUL"] = (
    full_train.groupby("engine_id")["cycle"]
    .transform("max") - full_train["cycle"]
)

# Generate historical features
train_features = engineer_features(
    full_train,
    active_sensors
)

test_features = engineer_features(
    test_raw,
    active_sensors
)

X_train_final = train_features[feature_columns]
y_train_final = train_features["RUL"]

# Last available cycle for each test engine
test_last = (
    test_features
    .sort_values(["engine_id", "cycle"])
    .groupby("engine_id", sort=True)
    .tail(1)
    .sort_values("engine_id")
    .reset_index(drop=True)
)

X_test_final = test_last[feature_columns]

# NASA test RUL labels correspond to engine IDs 1–100
assert np.array_equal(
    test_last["engine_id"].to_numpy(),
    np.arange(1, len(rul_df) + 1)
)

y_test_final = rul_df["rul"].to_numpy()

assert len(X_test_final) == len(y_test_final) == 100
assert not X_train_final.isna().any().any()
assert not X_test_final.isna().any().any()

print("Training:", X_train_final.shape)
print("Testing:", X_test_final.shape)
print("Final data preparation successful!")


C:\Users\EHAAB ULLAH\Projects\AeroGuard-AI-Predictive-Maintenance\ml\src\aeroguard_ml\feature_engineering.py:30: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f"{sensor}_mean_{window}"] = (
C:\Users\EHAAB ULLAH\Projects\AeroGuard-AI-Predictive-Maintenance\ml\src\aeroguard_ml\feature_engineering.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result[f"{sensor}_std_{window}"] = (
C:\Users\EHAAB ULLAH\Projects\AeroGuard-AI-Predictive-Maintenance\ml\src\aeroguard_ml\feature_engineering.py:50: PerformanceWarning: DataFram

Training: (20631, 123)
Testing: (100, 123)
Final data preparation successful!


In [ ]:

final_model = RandomForestRegressor(
    n_estimators=160,
    max_depth=18,
    min_samples_leaf=5,
    max_features=0.8,
    random_state=42,
    n_jobs=-1
)

print("Training final RF_B model...")

final_model.fit(X_train_final, y_train_final)

print("Final model training completed!")

# Predict on official NASA test engines
test_predictions = np.maximum(
    final_model.predict(X_test_final),
    0
)

mae = mean_absolute_error(
    y_test_final,
    test_predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_test_final,
        test_predictions
    )
)

bias = np.mean(
    test_predictions - y_test_final
)

print("\n===== OFFICIAL NASA FD001 TEST RESULTS =====")
print(f"MAE:  {mae:.4f} cycles")
print(f"RMSE: {rmse:.4f} cycles")
print(f"Bias: {bias:+.4f} cycles")

results = pd.DataFrame({
    "engine_id": test_last["engine_id"].astype(int),
    "last_cycle": test_last["cycle"].astype(int),
    "actual_rul": y_test_final,
    "predicted_rul": test_predictions
})

results["absolute_error"] = (
    results["actual_rul"] -
    results["predicted_rul"]
).abs()

display(results.head(10))
